# AI-Based Link Analysis & Threat Detection
## Machine Learning Model Development Notebook

**Author:** Data Science Portfolio Project  
**Purpose:** Train and evaluate multiple ML classifiers on URL features to detect phishing, malicious, and suspicious links.

---

### What this notebook covers

1. Import libraries
2. Load the dataset
3. Inspect & understand the data
4. Clean the data
5. Exploratory Data Analysis (EDA)
6. Feature extraction from raw URLs
7. Feature distribution visualisation
8. Train / test split
9. Train multiple ML models
10. Evaluate and compare models
11. Confusion matrices
12. ROC curves
13. Feature importance
14. Select the final model
15. Save the model to disk
16. Test with sample predictions

> **Note:** This notebook mirrors the `src/` pipeline used by the Flask application.  
> Run `python -m src.train_model` from the project root to retrain without opening Jupyter.

---
## 1. Import Libraries

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')

# Add project root to path so we can import our src/ modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), '')) if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Notebook display settings
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 40)
plt.style.use('dark_background')
sns.set_palette('husl')

print('Libraries loaded successfully.')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

---
## 2. Load the Dataset

The dataset `data/urls.csv` contains labelled URLs with four categories:
- **safe** — legitimate websites
- **suspicious** — unusual patterns, URL shorteners, dubious domains
- **phishing** — credential-theft and impersonation attempts
- **malicious** — malware, exploits, command-and-control servers

In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'urls.csv')

df_raw = pd.read_csv(DATA_PATH)
print(f'Raw dataset shape : {df_raw.shape}')
print(f'Columns           : {list(df_raw.columns)}')
df_raw.head(10)

---
## 3. Inspect the Data

Before cleaning, we check for:
- Data types
- Missing values
- Label distribution
- Duplicate URLs

In [ ]:
print('=== Data Types ===')
print(df_raw.dtypes)
print()
print('=== Missing Values ===')
print(df_raw.isnull().sum())
print()
print('=== Label Distribution ===')
print(df_raw['label'].value_counts())
print()
print(f'Duplicate URLs: {df_raw["url"].duplicated().sum()}')

---
## 4. Clean the Dataset

Cleaning steps:
1. Drop rows with missing URL or label
2. Normalise labels to lowercase
3. Remove duplicate URLs
4. Keep only the four recognised label values

In [ ]:
VALID_LABELS = ['safe', 'suspicious', 'phishing', 'malicious']

df = df_raw.copy()
df.dropna(subset=['url', 'label'], inplace=True)
df['url']   = df['url'].str.strip()
df['label'] = df['label'].str.strip().str.lower()
df.drop_duplicates(subset=['url'], inplace=True)
df = df[df['label'].isin(VALID_LABELS)].reset_index(drop=True)

print(f'Clean dataset shape : {df.shape}')
print()
print('Label distribution after cleaning:')
print(df['label'].value_counts())

---
## 5. Exploratory Data Analysis (EDA)

Visualise the label distribution and basic URL characteristics before feature extraction.

In [ ]:
COLORS = {
    'safe':       '#16a34a',
    'suspicious': '#d97706',
    'phishing':   '#dc2626',
    'malicious':  '#7c3aed',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts = df['label'].value_counts().reindex(VALID_LABELS)
bar_colors = [COLORS[l] for l in counts.index]
bars = axes[0].bar(counts.index, counts.values, color=bar_colors, edgecolor='none', width=0.6)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(val), ha='center', va='bottom', fontsize=11, color='white')
axes[0].set_title('Label Distribution', fontsize=13)
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Category')

# Pie chart
axes[1].pie(counts.values, labels=[l.title() for l in counts.index],
            colors=bar_colors, autopct='%1.1f%%', startangle=90,
            textprops={'color': 'white'})
axes[1].set_title('Label Proportions', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# URL length distribution by label
df['url_length'] = df['url'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label in VALID_LABELS:
    subset = df[df['label'] == label]['url_length']
    axes[0].hist(subset, alpha=0.6, label=label.title(), color=COLORS[label], bins=25)

axes[0].set_title('URL Length Distribution by Category', fontsize=13)
axes[0].set_xlabel('URL Length (characters)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot
data_by_label = [df[df['label'] == l]['url_length'].values for l in VALID_LABELS]
bp = axes[1].boxplot(data_by_label, labels=[l.title() for l in VALID_LABELS],
                     patch_artist=True, notch=False)
for patch, label in zip(bp['boxes'], VALID_LABELS):
    patch.set_facecolor(COLORS[label])
    patch.set_alpha(0.7)

axes[1].set_title('URL Length Box Plot', fontsize=13)
axes[1].set_ylabel('URL Length')

plt.tight_layout()
plt.show()

print('Mean URL lengths by label:')
print(df.groupby('label')['url_length'].mean().round(1))

---
## 6. Feature Extraction

We use `src/feature_extraction.py` to derive 32 numeric features from each URL string.
These features capture:
- **Structural** properties (length, dots, slashes, hyphens, digits)
- **Protocol** properties (HTTPS vs HTTP)
- **Suspicious patterns** (IP address, @ symbol, hex chars, keywords, TLD)
- **Domain quality** (entropy, digit ratio)
- **Query parameters** (count, redirect params)

No network requests are made — all analysis is purely static.

In [ ]:
from src.feature_extraction import extract_features, get_feature_names

# Show features for a safe URL
sample_safe = 'https://www.google.com/search?q=python'
print(f'Sample URL: {sample_safe}')
print()
safe_features = extract_features(sample_safe)
for k, v in safe_features.items():
    print(f'  {k:<28} = {v}')

In [ ]:
# Compare safe vs phishing
sample_phishing = 'http://secure-paypal-login.malicious-site.xyz/verify?account=true'
phishing_features = extract_features(sample_phishing)

comparison = pd.DataFrame({
    'Feature':  list(safe_features.keys()),
    'Safe URL':     list(safe_features.values()),
    'Phishing URL': list(phishing_features.values()),
})
# Highlight differences
comparison['Different'] = comparison['Safe URL'] != comparison['Phishing URL']
print(f'Features that differ: {comparison["Different"].sum()} / {len(comparison)}')
comparison[comparison['Different']].set_index('Feature')[['Safe URL', 'Phishing URL']]

In [ ]:
# Extract features for all URLs
print('Extracting features for all URLs...')
feature_rows = []
for url in df['url']:
    try:
        feature_rows.append(extract_features(url))
    except Exception:
        feature_rows.append({name: 0 for name in get_feature_names()})

feature_names = get_feature_names()
X_df = pd.DataFrame(feature_rows)[feature_names]
X_df = X_df.fillna(0)

print(f'Feature matrix shape: {X_df.shape}')
X_df.head()

---
## 7. Feature Distribution Visualisation

Visualise how key features differ across categories.

In [ ]:
X_df_labelled = X_df.copy()
X_df_labelled['label'] = df['label'].values

# Binary feature prevalence by label
binary_features = [
    'uses_https', 'has_ip_address', 'has_at_symbol', 'has_hex_chars',
    'has_suspicious_tld', 'is_url_shortener', 'excessive_subdomains',
    'has_login_keyword', 'has_verify_keyword', 'has_redirect_param'
]

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(binary_features):
    pct = X_df_labelled.groupby('label')[feat].mean() * 100
    pct = pct.reindex(VALID_LABELS, fill_value=0)
    axes[i].bar(pct.index, pct.values, color=[COLORS[l] for l in pct.index])
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=9)
    axes[i].set_ylabel('% URLs')
    axes[i].set_ylim(0, 110)
    axes[i].tick_params(axis='x', labelsize=7, rotation=20)

plt.suptitle('Binary Feature Prevalence by URL Category (%)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(16, 12))
corr = X_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm',
            center=0, linewidths=0.3, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Train / Test Split & Label Encoding

We:
1. Encode string labels → integers (safe=0, suspicious=1, phishing=2, malicious=3)
2. Split 80% train / 20% test using stratification to preserve class ratios
3. Use `random_state=42` for reproducibility

In [ ]:
LABEL_TO_INT = {'safe': 0, 'suspicious': 1, 'phishing': 2, 'malicious': 3}
INT_TO_LABEL = {v: k for k, v in LABEL_TO_INT.items()}

X = X_df.values
y = df['label'].map(LABEL_TO_INT).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set   : {X_train.shape[0]} samples')
print(f'Test set       : {X_test.shape[0]} samples')
print(f'Feature count  : {X_train.shape[1]}')
print()

# Check class distribution
from collections import Counter
print('Train label distribution:', {INT_TO_LABEL[k]: v for k, v in sorted(Counter(y_train).items())})
print('Test  label distribution:', {INT_TO_LABEL[k]: v for k, v in sorted(Counter(y_test).items())})

---
## 9. Train Multiple ML Models

We train five classifiers:

| Model | Rationale |
|---|---|
| Logistic Regression | Fast linear baseline, good interpretability |
| Decision Tree | Fully explainable single-tree model |
| Random Forest | Ensemble of trees, robust to noise |
| Gradient Boosting | Strong sequential ensemble learner |
| XGBoost | Optimised gradient boosting (if available) |

For cybersecurity we use `class_weight='balanced'` where supported to handle class imbalance.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, random_state=42, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, random_state=42,
        class_weight='balanced', n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42
    ),
}

# Optionally add XGBoost
try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(
        n_estimators=150, learning_rate=0.1, max_depth=5,
        use_label_encoder=False, eval_metric='mlogloss', random_state=42
    )
    print('XGBoost available — included in comparison.')
except ImportError:
    print('XGBoost not installed — skipping.')

print(f'Models to train: {list(models.keys())}')

In [ ]:
trained_models = {}
results        = {}

CLASS_NAMES = [INT_TO_LABEL[i] for i in range(4)]

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    trained_models[name] = model

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    try:
        y_bin = label_binarize(y_test, classes=sorted(np.unique(y_test)))
        roc   = roc_auc_score(y_bin, y_proba[:, :y_bin.shape[1]],
                              average='macro', multi_class='ovr')
    except Exception:
        roc = float('nan')

    results[name] = {
        'accuracy': acc, 'precision': prec, 'recall': rec,
        'f1': f1, 'roc_auc': roc,
        'y_pred': y_pred, 'y_proba': y_proba,
    }
    print(f'  Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}')

print('\nAll models trained.')

---
## 10. Model Evaluation & Comparison

> **Why not just use accuracy?**  
> In cybersecurity, a high-accuracy model might still miss many threats (low recall) or produce many false alarms (low precision). We report all metrics and weight recall for threat detection tasks.

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model':     name,
        'Accuracy':  f"{m['accuracy']:.4f}",
        'Precision': f"{m['precision']:.4f}",
        'Recall':    f"{m['recall']:.4f}",
        'F1-Score':  f"{m['f1']:.4f}",
        'ROC-AUC':   f"{m['roc_auc']:.4f}" if not np.isnan(m['roc_auc']) else 'N/A',
    }
    for name, m in results.items()
])

print('=' * 70)
print('  MODEL COMPARISON REPORT')
print('=' * 70)
print(comparison_df.to_string(index=False))

In [ ]:
# Visual comparison bar chart
metrics    = ['accuracy', 'precision', 'recall', 'f1']
model_names = list(results.keys())
x          = np.arange(len(model_names))
width      = 0.18
bar_colors = ['#2563eb', '#16a34a', '#dc2626', '#d97706']

fig, ax = plt.subplots(figsize=(13, 6))
for i, metric in enumerate(metrics):
    vals = [results[m][metric] for m in model_names]
    ax.bar(x + i * width, vals, width, label=metric.capitalize(),
           color=bar_colors[i], alpha=0.9)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, rotation=12, ha='right')
ax.set_ylim(0, 1.12)
ax.set_title('Model Comparison — Classification Metrics', fontsize=13)
ax.set_ylabel('Score')
ax.legend(loc='lower right')
ax.axhline(0.9, color='white', linestyle='--', linewidth=0.7, alpha=0.4)
plt.tight_layout()
plt.show()

---
## 11. Confusion Matrices

A confusion matrix shows exactly which classes are being confused with which. For threat detection, we care most about **false negatives** (a threat predicted as safe).

In [ ]:
n_models = len(results)
cols     = min(3, n_models)
rows     = (n_models + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
axes = np.array(axes).flatten()

for i, (name, m) in enumerate(results.items()):
    cm = confusion_matrix(y_test, m['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=axes[i], cbar=False)
    axes[i].set_title(f'{name}\n(F1={m["f1"]:.4f})', fontsize=11)
    axes[i].set_ylabel('True')
    axes[i].set_xlabel('Predicted')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Confusion Matrices — All Models', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 12. ROC Curves

ROC curves show the trade-off between the true positive rate and false positive rate across classification thresholds. Higher AUC = better discrimination.

In [ ]:
all_classes = sorted(np.unique(y_test))
n_classes   = len(all_classes)
y_test_bin  = label_binarize(y_test, classes=all_classes)

fig, axes = plt.subplots(1, n_classes, figsize=(n_classes * 5, 5))

line_colors = ['#2563eb', '#16a34a', '#dc2626', '#d97706', '#06b6d4']

for ci, cls_idx in enumerate(all_classes):
    ax = axes[ci]
    ax.plot([0, 1], [0, 1], 'w--', alpha=0.4, lw=1)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=9)
    ax.set_ylabel('True Positive Rate', fontsize=9)
    ax.set_title(f'ROC — {INT_TO_LABEL[cls_idx].upper()}', fontsize=11)

    for mi, (name, m) in enumerate(results.items()):
        if m['y_proba'] is None:
            continue
        fpr, tpr, _ = roc_curve(y_test_bin[:, ci], m['y_proba'][:, ci])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=1.5, color=line_colors[mi % len(line_colors)],
                label=f'{name} (AUC={roc_auc:.3f})')

    ax.legend(fontsize=7, loc='lower right')

plt.suptitle('ROC Curves per Class (One-vs-Rest)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 13. Feature Importance

For tree-based models we can directly inspect which features drove the predictions most.

In [ ]:
tree_models = {name: m for name, m in trained_models.items()
               if hasattr(m, 'feature_importances_')}

fig, axes = plt.subplots(1, len(tree_models), figsize=(len(tree_models) * 8, 6))
if len(tree_models) == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, tree_models.items()):
    importances = model.feature_importances_
    top_n       = 15
    indices     = np.argsort(importances)[::-1][:top_n]
    top_feats   = [feature_names[i] for i in indices]
    top_imps    = importances[indices]

    ax.barh(top_feats[::-1], top_imps[::-1], color='#2563eb', alpha=0.85)
    ax.set_title(f'{name}\nTop {top_n} Features', fontsize=11)
    ax.set_xlabel('Importance Score')

plt.suptitle('Feature Importance Comparison', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 14. Select the Final Model

**Selection criteria (for cybersecurity threat detection):**

We do NOT simply pick the highest accuracy. We use a weighted composite score:
- **45%** F1-score (balance of precision + recall)
- **30%** ROC-AUC (ranking quality)
- **25%** Recall (catching real threats is the priority)

Missing a real threat (false negative) is more dangerous than a false alarm (false positive).

In [ ]:
scores = {}
for name, m in results.items():
    f1      = m['f1']      or 0
    roc_auc = m['roc_auc'] if not np.isnan(m['roc_auc']) else 0
    recall  = m['recall']  or 0
    score   = 0.45 * f1 + 0.30 * roc_auc + 0.25 * recall
    scores[name] = round(score, 4)
    print(f'{name:<25}  composite score = {score:.4f}  '
          f'(F1={f1:.4f}, AUC={roc_auc:.4f}, Recall={recall:.4f})')

best_name = max(scores, key=scores.get)
print(f'\n>>> Selected model: {best_name} (score={scores[best_name]:.4f})')

In [ ]:
best_model = trained_models[best_name]

print(f'Classification Report — {best_name}:')
print(classification_report(
    y_test, results[best_name]['y_pred'],
    target_names=CLASS_NAMES, zero_division=0
))

---
## 15. Save the Model

We save:
1. `models/link_detection_model.pkl` — the trained classifier
2. `models/feature_config.pkl` — the feature names and label mapping (ensures train/inference consistency)

In [ ]:
MODELS_DIR  = os.path.join(PROJECT_ROOT, 'models')
MODEL_PATH  = os.path.join(MODELS_DIR, 'link_detection_model.pkl')
CONFIG_PATH = os.path.join(MODELS_DIR, 'feature_config.pkl')

os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(best_model, MODEL_PATH)

feature_config = {
    'feature_names':  feature_names,
    'label_to_int':   LABEL_TO_INT,
    'int_to_label':   {str(k): v for k, v in INT_TO_LABEL.items()},
    'selected_model': best_name,
    'n_classes':      4,
    'class_names':    CLASS_NAMES,
    'train_size':     len(X_train),
    'test_size':      len(X_test),
}
joblib.dump(feature_config, CONFIG_PATH)

print(f'Model saved  → {MODEL_PATH}')
print(f'Config saved → {CONFIG_PATH}')

---
## 16. Test Sample Predictions

Verify the saved model produces sensible results on a few example URLs.

In [ ]:
from src.predict import predict

test_urls = [
    'https://www.google.com',
    'http://192.168.1.1/admin/login.php',
    'https://secure-paypal-verify.malicious-site.xyz/login',
    'http://malware-distribution.xyz/download/setup.exe',
    'http://bit.ly/suspicious-link',
    'https://www.wikipedia.org/wiki/Machine_learning',
]

print(f'{"URL":<60} {"Prediction":<12} {"Risk":<8} {"Confidence"}')
print('-' * 95)
for url in test_urls:
    result = predict(url)
    print(f'{url[:58]:<60} {result["prediction"]:<12} {result["risk_score"]:<8} {result["confidence"]*100:.1f}%')

In [ ]:
# Detailed view of one result
result = predict('https://secure-paypal-verify.malicious-site.xyz/login')
print(f'URL        : {result["url"]}')
print(f'Prediction : {result["prediction"]}')
print(f'Risk Score : {result["risk_score"]}/100  ({result["risk_level"]})')
print(f'Confidence : {result["confidence"]*100:.1f}%')
print()
print('Why flagged:')
for flag in result['explanation']:
    print(f'  • {flag}')

---
## Summary

| Step | Outcome |
|---|---|
| Dataset | Labelled URL CSV with safe / suspicious / phishing / malicious categories |
| Features | 32 static URL features (no network requests) |
| Models trained | Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost |
| Selection criteria | Weighted composite of F1, ROC-AUC, and Recall |
| Output | Prediction, risk score (0-100), confidence, explanation flags |

### Limitations
- The model uses static URL features only — it cannot detect threats that depend on page content.
- The training dataset is small; a production system would use millions of samples.
- Model predictions are probabilistic, not guaranteed.
- Always verify suspicious URLs with additional tools before visiting them.

### Next Steps
- Collect a larger, more diverse dataset
- Add NLP features (URL string n-grams)
- Integrate real-time threat intelligence APIs
- Implement continuous retraining pipeline
- Add WHOIS / domain-age features